# Getting Started with Prepix

**Prepix** is a lightweight, reliable Python library for **automated data quality assessment, exploratory data analysis (EDA), and data contract validation**.

In this notebook, you will learn how to:
1. Profile column data types, missingness, and memory usage
2. Audit missing values and receive heuristic imputation suggestions
3. Detect and extract duplicate records
4. Analyze column cardinality and detect unique identifiers
5. Generate comprehensive numerical distribution statistics
6. Produce a unified data health report (`pp.report`)
7. Validate data contracts and schemas (`pp.schema_report`)


In [1]:
import pandas as pd
import numpy as np
import prepix as pp

print(f'Prepix version: {pp.__version__}')


Prepix version: 0.1.0


## 1. Load or Define Sample Data

We start with a realistic e-commerce transactions DataFrame featuring missing values, duplicate records, mixed types, and outliers.


In [2]:
df = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005, 1005, 1006],
    "customer": ["Alice", "Bob", "Charlie", None, "Eva", "Eva", "Frank"],
    "amount": [120.50, 45.00, 310.00, 15.20, 89.00, 89.00, 2500.00],
    "items_count": [2, 1, 5, 1, 3, 3, 12],
    "channel": ["Web", "Mobile", "Web", "Store", "Mobile", "Mobile", "Web"],
    "is_discounted": [False, True, False, True, False, False, True]
})
df


,order_id,customer,amount,items_count,channel,is_discounted
0,1001,Alice,120.5,2,Web,False
1,1002,Bob,45.0,1,Mobile,True
2,1003,Charlie,310.0,5,Web,False
3,1004,NaN,15.2,1,Store,True
4,1005,Eva,89.0,3,Mobile,False
5,1005,Eva,89.0,3,Mobile,False
6,1006,Frank,2500.0,12,Web,True


## 2. Structural Column Profiling (`pp.column_summary`)

`column_summary()` returns data types, missing counts, uniqueness counts, deep memory usage, and heuristic suggested types without altering the input.


In [3]:
summary = pp.column_summary(df)
summary


,Column,Data Type,Missing Values,Missing Percentage(%),Unique Values,Memory(Bytes),Suggested Type
0,order_id,int64,0,0.00,6,56,Numeric
1,customer,str,1,14.29,5,400,Categorical
2,amount,float64,0,0.00,6,56,Numeric
3,items_count,int64,0,0.00,5,56,Numeric
4,channel,str,0,0.00,3,431,Categorical
5,is_discounted,bool,0,0.00,2,7,Binary


## 3. Missing Value Analysis (`pp.missing_report`)

`missing_report()` ranks columns by missingness percentage, assigns severity levels (`Low`, `Medium`, `High`), and provides context-aware imputation recommendations.


In [4]:
missing = pp.missing_report(df, only_missing=False)
missing


,Column,Missing Values,Missing Percentage(%),Status,Suggestion
0,customer,1,14.29,Medium,Consider Mode Imputation
1,order_id,0,0.00,No Missing,No Action Needed
2,amount,0,0.00,No Missing,No Action Needed
3,items_count,0,0.00,No Missing,No Action Needed
4,channel,0,0.00,No Missing,No Action Needed
5,is_discounted,0,0.00,No Missing,No Action Needed


## 4. Duplicate Record Detection (`pp.duplicate_report`)

`duplicate_report()` identifies duplicate rows. Setting `return_records=True` extracts the actual duplicate rows for direct inspection.


In [5]:
dup_stats = pp.duplicate_report(df)
dup_stats


,Total Rows,Duplicate Rows,Duplicate Percentage(%),Unique Rows,Status,Suggestion
0,7,1,14.29,6,Duplicates Found,Consider reviewing and removing duplicate records


In [6]:
# Extract the duplicate records directly
dup_records = pp.duplicate_report(df, return_records=True)
dup_records


,order_id,customer,amount,items_count,channel,is_discounted
5,1005,Eva,89.0,3,Mobile,False


## 5. Uniqueness & Cardinality Classification (`pp.unique_report`)

`unique_report()` classifies each column into one of six cardinality tiers (`Constant`, `Low`, `Medium`, `High`, `Very High`, `Very High (Unique)`) to detect candidate primary keys and zero-variance features.


In [7]:
unique = pp.unique_report(df)
unique


,Column,Data Type,Total Values,Unique Values,Unique Percentage(%),Duplicate Values,Cardinality,Status,Suggestion
0,order_id,int64,7,6,85.71,1,High,High Diversity,High uniqueness discrete values; review whethe...
1,customer,str,7,6,85.71,1,High,High Diversity,High cardinality categorical; review before on...
2,amount,float64,7,6,85.71,1,High,High Diversity,High uniqueness numeric distribution; review d...
3,items_count,int64,7,5,71.43,2,High,High Diversity,High uniqueness discrete numeric values; revie...
4,channel,str,7,3,42.86,4,Medium,Moderate Diversity,Suitable for standard encoding or analysis
5,is_discounted,bool,7,2,28.57,5,Medium,Moderate Diversity,Suitable for standard encoding or analysis


## 6. Numerical Diagnostics (`pp.numeric_summary`)

`numeric_summary()` computes 5-number summaries, sample standard deviation, and optional extended diagnostics (`IQR`, `Skewness`, `Kurtosis`, zero count, negative count, and Tukey 1.5x IQR outliers).


In [8]:
num_stats = pp.numeric_summary(df, extended=True)
num_stats


,Column,Data Type,Count,Missing Values,Mean,Median,Std,Min,Q1,Q3,Max,Range,IQR,Skewness,Kurtosis,Zeros,Negatives,Outliers
0,order_id,int64,7,0,1003.714286,1004.0,1.799471,1001.0,1002.5,1005.00,1006.0,5.0,2.50,-0.367755,-1.129931,0,0,0
1,amount,float64,7,0,452.671429,89.0,907.775517,15.2,67.0,215.25,2500.0,2484.8,148.25,2.587135,6.750621,0,0,1
2,items_count,int64,7,0,3.857143,3.0,3.848314,1.0,1.5,4.00,12.0,11.0,2.50,1.994275,4.269565,0,0,1


## 7. Unified Dataset Quality Report (`pp.report`)

`pp.report(df)` generates an all-in-one `PrepixReport` object synthesizing dataset overview metrics, prioritized data quality issues, and actionable recommendations.


In [9]:
rep = pp.report(df)
print(rep)


                    PREPIX QUALITY REPORT                    

1. DATASET OVERVIEW
--------------------------------------------------------------
  - Rows                        : 7
  - Columns                     : 6
  - Total Cells                 : 42
  - Memory Usage                : 1.11 KB
  - Numeric Columns             : 3
  - Categorical Columns         : 2
  - Datetime Columns            : 0
  - Boolean Columns             : 1
  - Other Columns               : 0
  - Total Missing Cells         : 1 (2.38%)
  - Columns with Missing        : 1
  - Duplicate Rows              : 1 (14.29%)

2. KEY FINDINGS & ISSUES
--------------------------------------------------------------
  [MEDIUM] Dataset: Duplicate Records - 1 duplicate rows (14.29%) detected
  [MEDIUM] Column 'customer': Moderate Missingness - 1 missing values (14.29%)
  [LOW   ] Column 'amount': Potential Outliers - 1 values outside 1.5x IQR boundaries
  [LOW   ] Column 'amount': Highly Skewed Distribution - Skewness of 

### Accessing Individual Report Sections & JSON Export


In [10]:
print('Overview Table:')
display(rep.overview)
print('\nPrioritized Issues:')
display(rep.issues)


Overview Table:


name 'display' is not defined

In [11]:
# Export report as JSON string or file
json_output = rep.to_json()
print('JSON Preview (first 250 characters):\n', json_output[:250], '...')


JSON Preview (first 250 characters):
 {
  "overview": [
    {
      "Metric": "Rows",
      "Value": "7"
    },
    {
      "Metric": "Columns",
      "Value": "6"
    },
    {
      "Metric": "Total Cells",
      "Value": "42"
    },
    {
      "Metric": "Memory Usage",
      "Value":  ...


## 8. Data Contract Validation (`pp.schema_report`)

`schema_report()` verifies that incoming data satisfies your expected schema, including required columns, expected data types, nullability, and uniqueness.


In [12]:
schema = {
    "order_id": {"type": "integer", "nullable": False, "unique": True},
    "customer": "string",
    "amount": {"type": "numeric", "nullable": False},
    "items_count": "integer",
    "channel": "string",
    "is_discounted": "boolean"
}

validation = pp.schema_report(df, schema=schema, allow_unexpected=True)
validation


,Column,Check,Expected,Actual,Status,Message
0,order_id,column_presence,present,present,PASS,Required column 'order_id' is present.
1,order_id,data_type,integer,int64,PASS,Data type matches expected 'integer'.
2,order_id,non_null,no_missing,0 missing,PASS,Column 'order_id' contains no missing values.
3,order_id,unique,unique,1 duplicate(s),FAIL,Column 'order_id' contains 1 duplicate value(s...
4,customer,column_presence,present,present,PASS,Required column 'customer' is present.
5,customer,data_type,string,str,PASS,Data type matches expected 'string'.
6,amount,column_presence,present,present,PASS,Required column 'amount' is present.
7,amount,data_type,numeric,float64,PASS,Data type matches expected 'numeric'.
8,amount,non_null,no_missing,0 missing,PASS,Column 'amount' contains no missing values.
9,items_count,column_presence,present,present,PASS,Required column 'items_count' is present.
